# Phase W3 — Geometric Analysis of Learned Wildfire Metrics

**HAMTools-exclusive capabilities** — none of these are possible with Gahtan et al.'s eikonal approach:

1. **Fire corridor geodesics** — a fan of geodesics from the ignition point, colored by predicted arrival time. Reveals directional anisotropy of the learned metric.
2. **Jacobi divergence field** — rate at which neighboring geodesics separate. High divergence = unstable spread corridor (e.g., wind channels between ridges).
3. **Flag curvature anomaly map** — Finsler flag curvature sampled on a spatial grid. Identifies terrain/fuel features that abruptly change fire dynamics.

**Prerequisites:** A trained W1 metric checkpoint, or set `USE_SYNTHETIC = True` to train a quick synthetic metric in-place.

**Reference:** Gahtan, Shpund & Bronstein (2026). arXiv:2603.00035.

---

## Runtime guide

| Mode | ~Time (T4 GPU) |
|------|----------------|
| `USE_SYNTHETIC = True`, `QUICK = True` | ~10 min |
| `USE_SYNTHETIC = True`, `QUICK = False` | ~40 min |
| `USE_SYNTHETIC = False` (load W1 checkpoint) | < 2 min (inference only) |

In [ ]:
# ============================================================
# CELL 1 — Colab environment setup
# ============================================================
import subprocess


def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:])
    return r.returncode == 0


_run("pip install -q --upgrade 'jax[cuda12]'")
_run("pip install -q git+https://github.com/hubibala/ham.git")
_run("pip install -q 'Pillow>=9.0' 'rasterio>=1.3'")
print("Installation complete. Restart runtime if JAX was upgraded.")

In [ ]:
# ============================================================
# CELL 2 — GPU configuration
# ============================================================
import jax

try:
    from ham.utils import configure_device
except ImportError:

    def configure_device(device: str):
        dev = jax.devices(device)[0]
        jax.config.update("jax_default_device", dev)
        print(f"[HAMTools inline] Default JAX device set to: {dev}")
        return dev


DEVICE = "gpu" if any("gpu" in str(d).lower() for d in jax.devices()) else "cpu"
configure_device(DEVICE)
print(f"Devices: {jax.devices()}  →  using {DEVICE}")

In [ ]:
# ============================================================
# CELL 3 — Configuration
# ============================================================
USE_SYNTHETIC = True  # False = load checkpoint from W1 run
QUICK = True  # only used when USE_SYNTHETIC=True
N_DIRECTIONS = 32  # geodesic fan directions (64 for publication quality)
N_STEPS = 15  # AVBD steps per geodesic
COMPUTE_CURVATURE = True  # set False to skip (expensive on large grids)
OUTPUT_DIR = "results/phaseW3"

# If USE_SYNTHETIC = False, configure these:
# CHECKPOINT_PATH = '/content/drive/MyDrive/ham_checkpoints/metric_w1_seed42.eqx'
# DATA_ROOT       = '/content/drive/MyDrive/sim2real_fire'
# SCENE_ID        = '0014_00426'
# EVENT_ID        = '0014_00001'

import os

os.makedirs(f"{OUTPUT_DIR}/figs", exist_ok=True)
print(
    f"Config: USE_SYNTHETIC={USE_SYNTHETIC}, N_DIRECTIONS={N_DIRECTIONS}, DEVICE={DEVICE}"
)

In [ ]:
# ============================================================
# CELL 4 — Imports
# ============================================================
import sys
import time

import matplotlib
import numpy as np

matplotlib.use("Agg")
import equinox as eqx
import jax
import matplotlib.pyplot as plt
from jax import config

config.update("jax_enable_x64", True)

sys.path.insert(0, "../")  # adjust to '/content/HAM/examples' on Colab
try:
    from experiment_wildfire_flat import (
        _ignition_to_world,
        bind_scenario_to_metric,
        get_config,
        make_metric,
        make_solver,
    )
    from experiment_wildfire_geometric import (
        _make_and_train_synthetic_metric,
        _make_synthetic_scenario,
        compute_curvature_map,
        compute_jacobi_divergence,
        run_geometric_analysis,
        shoot_geodesic_fan,
    )

    print("Imports OK")
except ImportError as e:
    print(f"Import error: {e}")
    print("Make sure examples/ is in sys.path.")

In [ ]:
# ============================================================
# CELL 5 — Train or load metric
# ============================================================
cfg = get_config(quick=QUICK)

if USE_SYNTHETIC:
    print("Building synthetic scenario and training a quick W1 metric...")
    print(
        "(Set USE_SYNTHETIC=False and CHECKPOINT_PATH to use your trained W1 model)\n"
    )

    scenario = _make_synthetic_scenario()
    metric, solver, source_world, domain_shape = _make_and_train_synthetic_metric(
        scenario, n_epochs=cfg["n_epochs"]
    )

    print("\nTrained on 20×20 synthetic grid")
    print(f"Source (ignition) world coords: {source_world}")
    print(f"Domain size: {domain_shape:.1f} m")

else:
    from ham.geometry.manifolds import EuclideanSpace

    key = jax.random.PRNGKey(42)
    manifold = EuclideanSpace(2)
    metric_template = make_metric(cfg, manifold, key)
    metric = eqx.tree_deserialise_leaves(CHECKPOINT_PATH, metric_template)
    print(f"Loaded checkpoint: {CHECKPOINT_PATH}")

    solver = make_solver(cfg)

    # Load the corresponding scenario for context
    # (replace with your data loader if USE_SYNTHETIC=False)
    scenario = _make_synthetic_scenario()
    source_world = _ignition_to_world(scenario.ignition_pixel, scenario.pixel_spacing_m)
    domain_shape = float(scenario.arrival_times.shape[0]) * float(
        scenario.pixel_spacing_m
    )
    print(f"Source: {source_world},  domain ≈ {domain_shape:.0f} m")

In [ ]:
# ============================================================
# CELL 6 — Shoot geodesic fan from ignition point
# ============================================================
print(f"Shooting {N_DIRECTIONS} geodesics from ignition point {source_world}...")
t0 = time.time()
paths, arc_lengths = shoot_geodesic_fan(
    metric,
    solver,
    source_world,
    domain_size=float(domain_shape),
    n_directions=N_DIRECTIONS,
    n_steps=N_STEPS,
)
print(f"Done in {time.time() - t0:.1f}s")
print(
    f"Arc lengths range: [{float(arc_lengths.min()):.3f}, {float(arc_lengths.max()):.3f}]"
)

fig, ax = plt.subplots(figsize=(7, 7))
norm = plt.Normalize(float(arc_lengths.min()), float(arc_lengths.max()))
cmap = plt.cm.plasma

for path, arc in zip(paths, arc_lengths):
    ax.plot(
        path[:, 0], path[:, 1], color=cmap(norm(float(arc))), linewidth=1.2, alpha=0.85
    )

ax.scatter(
    *source_world, s=120, c="white", edgecolors="black", zorder=5, label="Ignition"
)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
plt.colorbar(sm, ax=ax, label="Predicted arc length (arrival order)")
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_title(f"Phase W3 — Fire corridor geodesics ({N_DIRECTIONS} directions)")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figs/w3_geodesic_fan.png", dpi=150)
plt.show()
print(f"Saved: {OUTPUT_DIR}/figs/w3_geodesic_fan.png")

In [ ]:
# ============================================================
# CELL 7 — Jacobi divergence field
# ============================================================
print("Computing Jacobi divergence along each geodesic path...")
t0 = time.time()
divergences = compute_jacobi_divergence(paths)  # (n_directions, n_steps)
print(f"Done in {time.time() - t0:.1f}s  — shape: {divergences.shape}")

mid_div = divergences[:, divergences.shape[1] // 2]
norm_div = plt.Normalize(np.percentile(mid_div, 5), np.percentile(mid_div, 95))
cmap_div = plt.cm.RdYlGn_r  # red = high divergence (unstable)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: fan colored by mid-path divergence
for path, div in zip(paths, mid_div):
    axes[0].plot(
        path[:, 0],
        path[:, 1],
        color=cmap_div(norm_div(float(div))),
        linewidth=1.4,
        alpha=0.85,
    )
axes[0].scatter(*source_world, s=100, c="white", edgecolors="black", zorder=5)
sm = plt.cm.ScalarMappable(cmap=cmap_div, norm=norm_div)
plt.colorbar(sm, ax=axes[0], label="Jacobi divergence (mid-path)")
axes[0].set_title("Geodesic stability (red = unstable corridor)")
axes[0].set_aspect("equal")
axes[0].set_xlabel("X (m)")
axes[0].set_ylabel("Y (m)")

# Right: divergence vs direction angle
angles_deg = np.linspace(0, 360, N_DIRECTIONS, endpoint=False)
mean_div = np.array(divergences).mean(axis=1)
axes[1].fill_between(angles_deg, mean_div, alpha=0.45, color="#EE7733")
axes[1].plot(angles_deg, mean_div, color="#CC4400", linewidth=1.5)
axes[1].set_xlabel("Direction (°)")
axes[1].set_ylabel("Jacobi divergence")
axes[1].set_title("Angular divergence profile — directional anisotropy")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figs/w3_jacobi_divergence.png", dpi=150)
plt.show()
print(f"Saved: {OUTPUT_DIR}/figs/w3_jacobi_divergence.png")

In [ ]:
# ============================================================
# CELL 8 — Flag curvature map
# ============================================================
if COMPUTE_CURVATURE:
    print("Computing flag curvature map on grid...")
    print("(Set COMPUTE_CURVATURE=False to skip — takes ~2–5 min on T4)")

    bound = bind_scenario_to_metric(metric, scenario)
    bound = bound.precompute_metric_field()

    curv_map, grid_x, grid_y = compute_curvature_map(
        bound,
        domain_size=float(domain_shape),
        n_grid=15,  # increase to 25+ for publication quality
        n_directions=8,
    )
    curv_map = np.array(curv_map)
    grid_x = np.array(grid_x)
    grid_y = np.array(grid_y)
    print(f"Curvature range: [{curv_map.min():.3f}, {curv_map.max():.3f}]")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    im = axes[0].contourf(grid_x, grid_y, curv_map, levels=20, cmap="coolwarm")
    plt.colorbar(im, ax=axes[0], label="Mean flag curvature K")
    axes[0].scatter(
        *source_world, s=120, c="black", marker="*", zorder=5, label="Ignition"
    )
    axes[0].set_title("Finsler flag curvature anomaly map")
    axes[0].set_xlabel("X (m)")
    axes[0].set_ylabel("Y (m)")
    axes[0].legend()
    axes[0].set_aspect("equal")

    for path in paths[:: max(1, N_DIRECTIONS // 8)]:
        axes[1].plot(path[:, 0], path[:, 1], color="black", linewidth=0.8, alpha=0.5)
    im2 = axes[1].contourf(
        grid_x, grid_y, curv_map, levels=15, cmap="coolwarm", alpha=0.6
    )
    plt.colorbar(im2, ax=axes[1], label="Flag curvature K")
    axes[1].scatter(
        *source_world, s=120, c="white", marker="*", edgecolors="black", zorder=6
    )
    axes[1].set_title("Geodesics on curvature background")
    axes[1].set_aspect("equal")

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/figs/w3_curvature_map.png", dpi=150)
    plt.show()
    print(f"Saved: {OUTPUT_DIR}/figs/w3_curvature_map.png")
else:
    print("Curvature computation skipped (COMPUTE_CURVATURE=False).")
    curv_map = grid_x = grid_y = None

In [ ]:
# ============================================================
# CELL 9 — Full geometric analysis pipeline
# ============================================================
print("Running full geometric analysis pipeline...")
run_geometric_analysis(
    metric=metric,
    solver=solver,
    source_world=source_world,
    scenario=scenario,
    domain_shape=domain_shape,
    output_dir=f"{OUTPUT_DIR}/figs",
    suffix="_notebook",
    n_directions=N_DIRECTIONS,
    n_steps=N_STEPS,
    compute_curvature=COMPUTE_CURVATURE,
)
print(f"\nAll figures saved to {OUTPUT_DIR}/figs/")

In [ ]:
# ============================================================
# CELL 10 — 3-panel publication summary figure
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel (a): geodesic fan
norm = plt.Normalize(float(arc_lengths.min()), float(arc_lengths.max()))
for path, arc in zip(paths, arc_lengths):
    axes[0].plot(
        path[:, 0],
        path[:, 1],
        color=plt.cm.plasma(norm(float(arc))),
        linewidth=1.0,
        alpha=0.8,
    )
axes[0].scatter(*source_world, s=100, c="white", edgecolors="k", zorder=5)
axes[0].set_title("(a) Fire corridor geodesics")
axes[0].set_aspect("equal")
axes[0].axis("off")

# Panel (b): Jacobi divergence
norm_div2 = plt.Normalize(np.percentile(mid_div, 5), np.percentile(mid_div, 95))
for path, div in zip(paths, mid_div):
    axes[1].plot(
        path[:, 0],
        path[:, 1],
        color=plt.cm.RdYlGn_r(norm_div2(float(div))),
        linewidth=1.0,
        alpha=0.8,
    )
axes[1].scatter(*source_world, s=100, c="white", edgecolors="k", zorder=5)
axes[1].set_title("(b) Jacobi divergence  (red = unstable)")
axes[1].set_aspect("equal")
axes[1].axis("off")

# Panel (c): flag curvature
if COMPUTE_CURVATURE and curv_map is not None:
    cf = axes[2].contourf(grid_x, grid_y, curv_map, levels=12, cmap="coolwarm")
    for path in paths[:: max(1, N_DIRECTIONS // 8)]:
        axes[2].plot(path[:, 0], path[:, 1], "k-", linewidth=0.6, alpha=0.35)
    axes[2].scatter(
        *source_world, s=100, c="white", marker="*", edgecolors="k", zorder=6
    )
    axes[2].set_title("(c) Flag curvature anomaly map")
    axes[2].set_aspect("equal")
    axes[2].axis("off")
    plt.colorbar(cf, ax=axes[2], shrink=0.7, label="K")
else:
    axes[2].text(
        0.5,
        0.5,
        "Curvature\nskipped",
        ha="center",
        va="center",
        transform=axes[2].transAxes,
        fontsize=13,
    )
    axes[2].axis("off")

plt.suptitle(
    "Phase W3 — Geometric Analysis of Learned Wildfire Metric (HAMTools-exclusive)",
    fontsize=11,
)
plt.tight_layout()
SUMMARY_PATH = f"{OUTPUT_DIR}/figs/w3_summary_panel.png"
plt.savefig(SUMMARY_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"Publication summary panel saved: {SUMMARY_PATH}")

# Download from Colab:
# from google.colab import files; files.download(SUMMARY_PATH)